# 01 — Data Audit and Exploratory Data Analysis

## Objectives

This notebook will:

1. Verify the dataset's real dimensions and schema.
2. Investigate possible Excel export truncation.
3. Measure timestamp and city coverage.
4. Detect missing values, duplicates, invalid values, and temporal gaps.
5. Determine whether seasonal analysis is defensible.
6. examine provisional AQI class balance.
7. Recommend Dhaka plus 2–4 cities for the project.

No preprocessing or modeling is performed in this notebook.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
from IPython.display import display

import hashlib
import json
import os
import re
import subprocess
import unicodedata
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

REPO_NAME = "cse437-air-quality-group-18"
REPO_URL = "https://github.com/ArafUlHaque/cse437-air-quality-group-18.git"
GIT_BRANCH = "main"

os.chdir("/content")
REPO_DIR = Path("/content") / REPO_NAME

if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", GIT_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

STORAGE_ROOT = Path("/content/drive/MyDrive/CSE437_air_quality_group_18")
RAW_DIR = STORAGE_ROOT / "raw"
PROCESSED_DIR = STORAGE_ROOT / "processed"
AUDIT_DIR = PROCESSED_DIR / "notebook_01_audit"
FIGURES_DIR = STORAGE_ROOT / "figures" / "notebook_01"

RAW_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Repository:", REPO_DIR)
print("Raw data:", RAW_DIR)
print("Audit outputs:", AUDIT_DIR)
print("Figures:", FIGURES_DIR)

Repository: /content/cse437-air-quality-group-18
Raw data: /content/drive/MyDrive/CSE437_air_quality_group_18/raw
Audit outputs: /content/drive/MyDrive/CSE437_air_quality_group_18/processed/notebook_01_audit
Figures: /content/drive/MyDrive/CSE437_air_quality_group_18/figures/notebook_01


In [3]:
RAW_FILE_NAME = ""  # Example: "bangladesh_air_quality.csv". Leave blank for automatic detection.

if RAW_FILE_NAME:
    RAW_FILE = RAW_DIR / RAW_FILE_NAME
    if not RAW_FILE.is_file():
        raise FileNotFoundError(f"File not found: {RAW_FILE}")
else:
    csv_files = sorted(RAW_DIR.glob("*.csv"))

    if len(csv_files) == 0:
        raise FileNotFoundError(f"No CSV file found in {RAW_DIR}")

    if len(csv_files) > 1:
        print("Multiple CSV files found:")
        for file in csv_files:
            print("-", file.name)
        raise ValueError("Set RAW_FILE_NAME to the correct source filename.")

    RAW_FILE = csv_files[0]

def calculate_sha256(file_path):
    digest = hashlib.sha256()

    with file_path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

file_size_mb = RAW_FILE.stat().st_size / (1024 ** 2)
file_checksum = calculate_sha256(RAW_FILE)

print("Source file:", RAW_FILE.name)
print(f"File size: {file_size_mb:,.2f} MB")
print("SHA-256:", file_checksum)

FileNotFoundError: No CSV file found in /content/drive/MyDrive/CSE437_air_quality_group_18/raw

In [ ]:
df_raw = pd.read_csv(RAW_FILE, low_memory=False)

print("Dataset loaded.")
print(f"Rows: {len(df_raw):,}")
print(f"Columns: {df_raw.shape[1]:,}")

display(df_raw.head())
display(df_raw.tail())

In [ ]:
print("Original column names:")
for index, column in enumerate(df_raw.columns, start=1):
    print(f"{index:>2}. {column}")

print()
df_raw.info()

In [ ]:
def normalize_column_name(column):
    column = unicodedata.normalize("NFKC", str(column)).lower()
    column = column.replace("2.5", "25")
    column = re.sub(r"[^a-z0-9]+", "_", column)
    return column.strip("_")

ALIASES = {
    "city_id": ["city_id", "location_id", "station_id"],
    "city": ["city", "city_name", "location", "location_name", "station", "station_name"],
    "latitude": ["latitude", "lat"],
    "longitude": ["longitude", "longitude_value", "lon", "lng"],
    "timestamp": ["timestamp", "date_time", "datetime", "time", "date"],
    "pm10": ["pm10", "pm_10"],
    "pm25": ["pm25", "pm_25", "pm2_5"],
    "co": ["co", "carbon_monoxide"],
    "co2": ["co2", "carbon_dioxide"],
    "no2": ["no2", "nitrogen_dioxide"],
    "so2": ["so2", "sulfur_dioxide", "sulphur_dioxide"],
    "o3": ["o3", "ozone"],
    "aqi": ["aqi", "air_quality_index"],
}

normalized_columns = {}

for column in df_raw.columns:
    normalized = normalize_column_name(column)

    if normalized in normalized_columns:
        raise ValueError(f"Column-name collision: {column} and {normalized_columns[normalized]}")

    normalized_columns[normalized] = column

column_mapping = {}

for canonical, aliases in ALIASES.items():
    for alias in aliases:
        normalized_alias = normalize_column_name(alias)

        if normalized_alias in normalized_columns:
            column_mapping[canonical] = normalized_columns[normalized_alias]
            break

mapping_table = pd.DataFrame({
    "canonical_name": list(ALIASES),
    "source_column": [column_mapping.get(column, "NOT FOUND") for column in ALIASES],
})

display(mapping_table)

required_columns = ["city", "timestamp", "aqi"]
missing_required = [column for column in required_columns if column not in column_mapping]

if missing_required:
    raise ValueError(f"Required columns not found: {missing_required}")

audit = df_raw.rename(columns={source: canonical for canonical, source in column_mapping.items()}).copy()

print("Required columns found successfully.")

In [ ]:
audit["city"] = audit["city"].astype("string").str.strip()
audit.loc[audit["city"].eq(""), "city"] = pd.NA

audit["timestamp_raw"] = audit["timestamp"].astype("string")

try:
    audit["timestamp"] = pd.to_datetime(audit["timestamp_raw"], errors="coerce", format="mixed", utc=True)
except ValueError:
    audit["timestamp"] = pd.to_datetime(audit["timestamp_raw"], errors="coerce", utc=True)

numeric_columns = [
    column
    for column in ["latitude", "longitude", "pm10", "pm25", "co", "co2", "no2", "so2", "o3", "aqi"]
    if column in audit.columns
]

numeric_parse_failures = {}

for column in numeric_columns:
    original_non_missing = audit[column].notna()
    converted = pd.to_numeric(audit[column], errors="coerce")

    numeric_parse_failures[column] = int((original_non_missing & converted.isna()).sum())
    audit[column] = converted

timestamp_examples = audit[["timestamp_raw", "timestamp"]].head(10)
display(timestamp_examples)

print("Unparseable timestamps:", audit["timestamp"].isna().sum())
print("Numeric parsing failures:")
display(pd.Series(numeric_parse_failures, name="parse_failures").to_frame())

In [ ]:
PUBLISHED_ROWS = 1_048_551
PUBLISHED_CITIES = 103
PUBLISHED_YEARS = 25

EXCEL_TOTAL_ROW_LIMIT = 1_048_576
EXCEL_DATA_ROW_LIMIT = EXCEL_TOTAL_ROW_LIMIT - 1

actual_rows = len(audit)
actual_cities = audit["city"].nunique(dropna=True)

theoretical_rows = int(PUBLISHED_CITIES * PUBLISHED_YEARS * 365.25 * 24)
equivalent_years_per_city = actual_rows / (max(actual_cities, 1) * 365.25 * 24)
distance_from_excel_limit = EXCEL_DATA_ROW_LIMIT - actual_rows

arithmetic_audit = pd.DataFrame({
    "check": [
        "Actual rows",
        "Published rows",
        "Actual cities",
        "Published cities",
        "Approximate rows for complete 25-year hourly coverage",
        "Observed rows as percentage of theoretical rows",
        "Equivalent complete years per observed city",
        "Maximum Excel data rows with one header",
        "Rows below Excel data-row limit",
    ],
    "value": [
        actual_rows,
        PUBLISHED_ROWS,
        actual_cities,
        PUBLISHED_CITIES,
        theoretical_rows,
        100 * actual_rows / theoretical_rows,
        equivalent_years_per_city,
        EXCEL_DATA_ROW_LIMIT,
        distance_from_excel_limit,
    ],
})

display(arithmetic_audit)

if abs(distance_from_excel_limit) <= 100:
    print("WARNING: The row count is extremely close to Excel's data-row limit.")
else:
    print("The row count is not within 100 rows of Excel's data-row limit.")

In [ ]:
exact_duplicate_rows = int(df_raw.duplicated().sum())

valid_city_time = audit.dropna(subset=["city", "timestamp"])
duplicate_city_timestamps = int(valid_city_time.duplicated(["city", "timestamp"]).sum())

quality_summary = pd.DataFrame({
    "check": [
        "Rows",
        "Columns",
        "Missing city values",
        "Unparseable timestamps",
        "Exact duplicate rows",
        "Duplicate city-timestamp pairs",
    ],
    "count": [
        len(audit),
        audit.shape[1],
        audit["city"].isna().sum(),
        audit["timestamp"].isna().sum(),
        exact_duplicate_rows,
        duplicate_city_timestamps,
    ],
})

display(quality_summary)

In [ ]:
missingness = pd.DataFrame({
    "dtype": audit.dtypes.astype(str),
    "missing_count": audit.isna().sum(),
    "missing_percent": audit.isna().mean().mul(100),
}).sort_values("missing_percent", ascending=False)

display(missingness)
missingness.to_csv(AUDIT_DIR / "overall_missingness.csv")

In [ ]:
range_checks = []

for column in ["pm10", "pm25", "co", "co2", "no2", "so2", "o3"]:
    if column in audit.columns:
        range_checks.append({
            "check": f"{column} below 0",
            "flagged_rows": int(audit[column].lt(0).sum()),
        })

range_checks.extend([
    {"check": "AQI below 0", "flagged_rows": int(audit["aqi"].lt(0).sum())},
    {"check": "AQI above 500", "flagged_rows": int(audit["aqi"].gt(500).sum())},
])

if "latitude" in audit.columns:
    range_checks.append({
        "check": "Latitude outside approximate Bangladesh bounds",
        "flagged_rows": int((audit["latitude"].notna() & ~audit["latitude"].between(20, 27)).sum()),
    })

if "longitude" in audit.columns:
    range_checks.append({
        "check": "Longitude outside approximate Bangladesh bounds",
        "flagged_rows": int((audit["longitude"].notna() & ~audit["longitude"].between(88, 93)).sum()),
    })

range_check_table = pd.DataFrame(range_checks)
display(range_check_table)

print("These are audit flags only. No rows are removed in Notebook 01.")

In [ ]:
audit_valid = audit.dropna(subset=["city", "timestamp"]).copy()

audit_valid["hour"] = audit_valid["timestamp"].dt.floor("h")
audit_valid["date"] = audit_valid["timestamp"].dt.tz_convert(None).dt.normalize()

coverage = (
    audit_valid.groupby("city")
    .agg(
        row_count=("timestamp", "size"),
        unique_timestamps=("timestamp", "nunique"),
        unique_hours=("hour", "nunique"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
        observed_days=("date", "nunique"),
    )
    .reset_index()
)

coverage["span_days"] = (
    (coverage["last_timestamp"] - coverage["first_timestamp"]).dt.total_seconds() / 86400
) + 1

coverage["expected_hours_in_span"] = coverage["span_days"] * 24
coverage["hourly_coverage_percent"] = (
    100 * coverage["unique_hours"] / coverage["expected_hours_in_span"]
)

city_day_counts = (
    audit_valid.groupby(["city", "date"])
    .agg(
        row_count=("timestamp", "size"),
        unique_hours=("hour", "nunique"),
    )
    .reset_index()
)

daily_coverage = (
    city_day_counts.groupby("city")
    .agg(
        median_rows_per_day=("row_count", "median"),
        median_unique_hours_per_day=("unique_hours", "median"),
        minimum_unique_hours_per_day=("unique_hours", "min"),
        maximum_unique_hours_per_day=("unique_hours", "max"),
    )
    .reset_index()
)

aqi_completeness = (
    audit_valid.groupby("city")["aqi"]
    .apply(lambda values: 100 * values.notna().mean())
    .rename("aqi_non_missing_percent")
    .reset_index()
)

coverage = coverage.merge(daily_coverage, on="city", how="left")
coverage = coverage.merge(aqi_completeness, on="city", how="left")
coverage = coverage.sort_values("row_count", ascending=False).reset_index(drop=True)

display(coverage.head(25))
coverage.to_csv(AUDIT_DIR / "city_coverage_summary.csv", index=False)

In [ ]:
print("Overall recorded timestamp range")
print("Minimum:", audit_valid["timestamp"].min())
print("Maximum:", audit_valid["timestamp"].max())
print("Duration:", audit_valid["timestamp"].max() - audit_valid["timestamp"].min())

print()
print("Cities:", audit_valid["city"].nunique())
print("City-days:", len(city_day_counts))

In [ ]:
audit_columns = [
    column
    for column in ["pm10", "pm25", "co", "co2", "no2", "so2", "o3", "aqi"]
    if column in audit_valid.columns
]

missingness_by_city = (
    audit_valid.groupby("city")[audit_columns]
    .agg(lambda values: 100 * values.isna().mean())
    .round(2)
)

display(missingness_by_city.head(20))
missingness_by_city.to_csv(AUDIT_DIR / "missingness_by_city.csv")

In [ ]:
gap_records = []

for city, city_data in audit_valid.groupby("city"):
    timestamps = city_data["timestamp"].drop_duplicates().sort_values()
    differences = timestamps.diff().dt.total_seconds().div(3600).dropna()

    gap_records.append({
        "city": city,
        "median_interval_hours": differences.median(),
        "gaps_over_1_5_hours": int(differences.gt(1.5).sum()),
        "gaps_over_24_hours": int(differences.gt(24).sum()),
        "maximum_gap_hours": differences.max(),
    })

gap_summary = pd.DataFrame(gap_records)
gap_summary = gap_summary.sort_values("maximum_gap_hours", ascending=False)

display(gap_summary.head(25))
gap_summary.to_csv(AUDIT_DIR / "city_gap_summary.csv", index=False)

In [ ]:
MIN_HOURS_PER_DAY = 18
MIN_MONTHLY_DAY_COVERAGE = 0.80
MIN_COMPLETE_YEARS = 2

city_day_counts["usable_day"] = city_day_counts["unique_hours"].ge(MIN_HOURS_PER_DAY)
city_day_counts["year_month"] = city_day_counts["date"].dt.to_period("M")

monthly_coverage = (
    city_day_counts.groupby(["city", "year_month"])
    .agg(
        observed_days=("date", "nunique"),
        usable_days=("usable_day", "sum"),
    )
    .reset_index()
)

monthly_coverage["expected_days"] = monthly_coverage["year_month"].dt.days_in_month
monthly_coverage["usable_day_fraction"] = (
    monthly_coverage["usable_days"] / monthly_coverage["expected_days"]
)

monthly_coverage["usable_month"] = monthly_coverage["usable_day_fraction"].ge(
    MIN_MONTHLY_DAY_COVERAGE
)

monthly_coverage["year"] = monthly_coverage["year_month"].dt.year

yearly_coverage = (
    monthly_coverage.groupby(["city", "year"])
    .agg(
        months_present=("year_month", "nunique"),
        usable_months=("usable_month", "sum"),
    )
    .reset_index()
)

yearly_coverage["complete_usable_year"] = yearly_coverage["usable_months"].eq(12)

seasonality_summary = (
    yearly_coverage.groupby("city")
    .agg(
        years_present=("year", "nunique"),
        complete_usable_years=("complete_usable_year", "sum"),
    )
    .reset_index()
)

seasonality_summary["seasonality_supported"] = (
    seasonality_summary["complete_usable_years"] >= MIN_COMPLETE_YEARS
)

seasonality_summary = seasonality_summary.sort_values(
    ["complete_usable_years", "years_present"],
    ascending=False,
)

display(seasonality_summary.head(25))

monthly_coverage.to_csv(AUDIT_DIR / "monthly_coverage.csv", index=False)
seasonality_summary.to_csv(AUDIT_DIR / "seasonality_summary.csv", index=False)

In [ ]:
supported_city_count = seasonality_summary["seasonality_supported"].sum()

print("Cities supporting strong seasonal analysis:", supported_city_count)
print("Total audited cities:", len(seasonality_summary))

if supported_city_count == 0:
    print("Decision: Remove seasonal variation as a primary research question.")
else:
    print("Seasonality may be retained only for cities marked True above.")

In [ ]:
top_20_coverage = coverage.head(20).sort_values("row_count")
top_missing = missingness.loc[missingness["missing_percent"] > 0].head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].barh(top_20_coverage["city"], top_20_coverage["row_count"])
axes[0].set_title("Top 20 Cities by Number of Rows")
axes[0].set_xlabel("Rows")
axes[0].set_ylabel("City")

axes[1].barh(top_missing.index[::-1], top_missing["missing_percent"][::-1])
axes[1].set_title("Overall Missingness")
axes[1].set_xlabel("Missing values (%)")
axes[1].set_ylabel("Column")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_coverage_and_missingness.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
timeline = coverage.head(30).sort_values("first_timestamp").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(13, 10))

for index, row in timeline.iterrows():
    ax.hlines(index, row["first_timestamp"], row["last_timestamp"], linewidth=4)

ax.set_yticks(range(len(timeline)))
ax.set_yticklabels(timeline["city"])
ax.set_title("Recorded Time Span of the 30 Largest City Subsets")
ax.set_xlabel("Timestamp")
ax.set_ylabel("City")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_city_coverage_timeline.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plot_columns = [
    column
    for column in ["pm10", "pm25", "co", "no2", "so2", "o3", "aqi"]
    if column in audit_valid.columns
]

sample_size = min(100_000, len(audit_valid))
eda_sample = audit_valid[plot_columns].sample(sample_size, random_state=RANDOM_SEED)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(eda_sample["aqi"].dropna(), bins=50, ax=axes[0])
axes[0].axvline(100, color="orange", linestyle="--", label="USG or worse: >100")
axes[0].axvline(150, color="red", linestyle="--", label="Unhealthy: >150")
axes[0].set_title("Hourly AQI Distribution")
axes[0].set_xlabel("AQI")
axes[0].legend()

correlation = eda_sample.corr(method="spearman")
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Spearman Correlation of Numeric Air-Quality Variables")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_aqi_distribution_and_correlation.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
daily_diagnostic = (
    audit_valid.groupby(["city", "date"])
    .agg(
        daily_aqi=("aqi", "max"),
        unique_hours=("hour", "nunique"),
        valid_aqi_hours=("aqi", "count"),
    )
    .reset_index()
)

usable_daily_diagnostic = daily_diagnostic.loc[
    daily_diagnostic["unique_hours"].ge(MIN_HOURS_PER_DAY)
    & daily_diagnostic["valid_aqi_hours"].ge(MIN_HOURS_PER_DAY)
].copy()

usable_daily_diagnostic["unhealthy_151_plus"] = usable_daily_diagnostic["daily_aqi"].gt(150)
usable_daily_diagnostic["usg_or_worse_101_plus"] = usable_daily_diagnostic["daily_aqi"].gt(100)

threshold_summary = (
    usable_daily_diagnostic.groupby("city")
    .agg(
        usable_days=("date", "nunique"),
        unhealthy_days_151_plus=("unhealthy_151_plus", "sum"),
        unhealthy_rate_151_plus=("unhealthy_151_plus", "mean"),
        usg_or_worse_days_101_plus=("usg_or_worse_101_plus", "sum"),
        usg_or_worse_rate_101_plus=("usg_or_worse_101_plus", "mean"),
    )
    .reset_index()
)

threshold_summary["unhealthy_rate_151_plus"] *= 100
threshold_summary["usg_or_worse_rate_101_plus"] *= 100

threshold_summary = threshold_summary.sort_values(
    "unhealthy_rate_151_plus",
    ascending=False,
)

display(threshold_summary.head(25))
threshold_summary.to_csv(AUDIT_DIR / "provisional_threshold_summary.csv", index=False)

In [ ]:
DHAKA_NAME = "Dhaka"

city_names = sorted(audit_valid["city"].dropna().unique())
city_lookup = {str(city).casefold(): city for city in city_names}

if DHAKA_NAME.casefold() not in city_lookup:
    possible_matches = [
        city for city in city_names
        if "dhaka" in str(city).casefold()
    ]

    print("Possible Dhaka matches:", possible_matches)
    raise ValueError("Update DHAKA_NAME to the exact dataset spelling.")

dhaka_city = city_lookup[DHAKA_NAME.casefold()]

usable_date_sets = {
    city: set(group["date"])
    for city, group in usable_daily_diagnostic.groupby("city")
}

dhaka_dates = usable_date_sets.get(dhaka_city, set())

selection_table = coverage.merge(gap_summary, on="city", how="left")
selection_table = selection_table.merge(seasonality_summary, on="city", how="left")
selection_table = selection_table.merge(threshold_summary, on="city", how="left")

selection_table["overlap_days_with_dhaka"] = selection_table["city"].map(
    lambda city: len(usable_date_sets.get(city, set()) & dhaka_dates)
)

MIN_OBSERVED_DAYS = 365
MIN_DHAKA_OVERLAP_DAYS = 365
MIN_MEDIAN_HOURS = 18
MIN_AQI_COMPLETENESS = 80

selection_table["eligible"] = (
    selection_table["observed_days"].ge(MIN_OBSERVED_DAYS)
    & selection_table["overlap_days_with_dhaka"].ge(MIN_DHAKA_OVERLAP_DAYS)
    & selection_table["median_unique_hours_per_day"].ge(MIN_MEDIAN_HOURS)
    & selection_table["aqi_non_missing_percent"].ge(MIN_AQI_COMPLETENESS)
)

selection_table = selection_table.sort_values(
    [
        "eligible",
        "overlap_days_with_dhaka",
        "hourly_coverage_percent",
        "aqi_non_missing_percent",
    ],
    ascending=False,
)

selection_columns = [
    "city",
    "eligible",
    "row_count",
    "observed_days",
    "overlap_days_with_dhaka",
    "median_unique_hours_per_day",
    "hourly_coverage_percent",
    "aqi_non_missing_percent",
    "complete_usable_years",
    "unhealthy_rate_151_plus",
]

display(selection_table[selection_columns].head(25))
selection_table.to_csv(AUDIT_DIR / "city_selection_candidates.csv", index=False)

In [ ]:
eligible_cities = selection_table.loc[
    selection_table["eligible"]
    & selection_table["city"].ne(dhaka_city),
    "city",
].tolist()

recommended_cities = [dhaka_city]
common_dates = set(dhaka_dates)

while len(recommended_cities) < 5 and eligible_cities:
    overlap_scores = {
        city: len(common_dates & usable_date_sets.get(city, set()))
        for city in eligible_cities
    }

    best_city = max(overlap_scores, key=overlap_scores.get)
    best_common_days = overlap_scores[best_city]

    if best_common_days < MIN_DHAKA_OVERLAP_DAYS:
        break

    recommended_cities.append(best_city)
    common_dates &= usable_date_sets[best_city]
    eligible_cities.remove(best_city)

print("Recommended cities:", recommended_cities)
print("Common usable dates:", len(common_dates))

if common_dates:
    print("First common date:", min(common_dates))
    print("Last common date:", max(common_dates))

if len(recommended_cities) < 3:
    print("WARNING: Fewer than three cities passed the current eligibility rules.")
    print("Review the thresholds, coverage, and faculty expectations before continuing.")

In [ ]:
recommended_daily = usable_daily_diagnostic.loc[
    usable_daily_diagnostic["city"].isin(recommended_cities)
].copy()

recommended_thresholds = threshold_summary.loc[
    threshold_summary["city"].isin(recommended_cities)
].copy()

fig, axes = plt.subplots(2, 1, figsize=(15, 12))

sns.lineplot(
    data=recommended_daily,
    x="date",
    y="daily_aqi",
    hue="city",
    linewidth=1,
    ax=axes[0],
)

axes[0].axhline(150, color="red", linestyle="--", label="Unhealthy threshold")
axes[0].set_title("Provisional Daily Maximum AQI")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Daily maximum AQI")

sns.barplot(
    data=recommended_thresholds,
    x="city",
    y="unhealthy_rate_151_plus",
    ax=axes[1],
)

axes[1].set_title("Provisional Unhealthy-Day Rate")
axes[1].set_xlabel("City")
axes[1].set_ylabel("Days with AQI > 150 (%)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_recommended_city_diagnostics.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
selected_seasonality = seasonality_summary.loc[
    seasonality_summary["city"].isin(recommended_cities)
]

seasonality_supported_for_all = bool(
    len(selected_seasonality) == len(recommended_cities)
    and selected_seasonality["seasonality_supported"].all()
)

audit_summary = {
    "source_file": RAW_FILE.name,
    "file_size_mb": round(file_size_mb, 2),
    "sha256": file_checksum,
    "rows": int(len(audit)),
    "columns": int(audit.shape[1]),
    "cities": int(actual_cities),
    "minimum_timestamp": str(audit_valid["timestamp"].min()),
    "maximum_timestamp": str(audit_valid["timestamp"].max()),
    "exact_duplicate_rows": exact_duplicate_rows,
    "duplicate_city_timestamp_pairs": duplicate_city_timestamps,
    "rows_below_excel_data_limit": int(distance_from_excel_limit),
    "equivalent_complete_years_per_city": round(equivalent_years_per_city, 3),
    "recommended_cities": recommended_cities,
    "common_usable_days": int(len(common_dates)),
    "common_first_date": str(min(common_dates)) if common_dates else None,
    "common_last_date": str(max(common_dates)) if common_dates else None,
    "seasonality_supported_for_all_recommended_cities": seasonality_supported_for_all,
    "provisional_daily_aqi_aggregation": "Maximum provided hourly AQI",
    "main_target": "Next calendar day's daily AQI > 150",
    "excluded_future_predictor": "CO2",
    "timestamp_note": "Parsed with utc=True for this audit; confirm the source timezone before final daily aggregation.",
}

with (AUDIT_DIR / "audit_summary.json").open("w", encoding="utf-8") as file:
    json.dump(audit_summary, file, indent=2)

print(json.dumps(audit_summary, indent=2))

In [ ]:
print("Saved audit files:")

for file in sorted(AUDIT_DIR.iterdir()):
    print("-", file.name)

print("\nSaved figures:")

for file in sorted(FIGURES_DIR.iterdir()):
    print("-", file.name)

## Audit-gate decision

### Verified dataset facts

- Actual rows:
- Actual columns:
- Number of cities:
- Overall minimum timestamp:
- Overall maximum timestamp:
- Distance from the Excel data-row limit:
- Evidence supporting or rejecting possible truncation:

### Selected scope

- Selected cities:
- Common study period:
- Common usable days:
- Reason each city was selected:

### Data-quality decisions

- Duplicate handling required:
- Invalid-value handling required:
- Missingness concerns:
- Timestamp/timezone interpretation:
- CO2 will be excluded: **Yes**

### Seasonality decision

- Number of complete usable years per selected city:
- Is seasonal analysis defensible?
- Final decision and justification:

### Target decision

- Daily AQI aggregation: proposed maximum provided hourly AQI
- Main positive class: next calendar day's daily AQI > 150
- Optional sensitivity analysis: next calendar day's daily AQI > 100
- Final approved definition:

### Audit gate

- [ ] Actual coverage verified
- [ ] Possible truncation investigated
- [ ] Three to five cities approved
- [ ] Common period approved
- [ ] Seasonal-analysis decision recorded
- [ ] Daily AQI definition approved
- [ ] Target threshold approved

**Notebook 02 must not begin until every item above is completed.**